# Manipulator polygon designer

Interactive editor for the swappable manipulator shapes attached to the diff-drive robot's chassis. Each manipulator is one or more **convex** polygons (≤ 8 vertices each) in body-local coordinates; the simulator attaches one Box2D shape per `part` to a single body, sharing the chassis pose.

**Conventions:** body frame, +x forward, meters. Chassis is drawn as a fixed gray square at the origin for reference.

**Controls:**
- Left-click empty space → add a vertex to the **active part**.
- Left-click + drag a vertex → move it.
- Right-click a vertex → delete it.
- Use the part dropdown to switch which polygon is active. New part → empty polygon to draw into. Delete part → removes the active one.
- Each part is auto-sorted CCW around its own centroid on every redraw.

**Output:** writes `sim/configs/manipulators/<name>.json` when you press Save.

In [ ]:
%matplotlib widget
import json
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import Polygon as MplPolygon, Rectangle

# Locate the AtomSim root (CMakePresets.json marker) and the configs dir.
here = Path.cwd().resolve()
atomsim_root = next((p for p in [here, *here.parents] if (p / "CMakePresets.json").exists()), None)
if atomsim_root is None:
    raise RuntimeError("Could not locate AtomSim/ from notebook cwd.")

CONFIGS_DIR = atomsim_root / "sim" / "configs" / "manipulators"
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

B2_MAX_VERTICES = 8
PART_COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red",
               "tab:purple", "tab:brown", "tab:pink", "tab:olive"]


In [ ]:
def sort_ccw(part):
    """Return the part's vertices sorted counter-clockwise around their centroid."""
    if len(part) < 3:
        return list(part)
    pts = np.asarray(part, dtype=float)
    c = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0])
    order = np.argsort(angles)
    return [tuple(pts[i]) for i in order]


def is_convex(part):
    """A simple polygon is convex iff all consecutive cross products share a sign."""
    if len(part) < 3:
        return False
    pts = np.asarray(part, dtype=float)
    n = len(pts)
    signs = []
    for i in range(n):
        a = pts[i]
        b = pts[(i + 1) % n]
        c = pts[(i + 2) % n]
        cross = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
        if abs(cross) < 1e-12:
            continue  # collinear — ignore
        signs.append(cross > 0)
    return len(signs) > 0 and (all(signs) or not any(signs))


def part_status(part):
    """Return (ok, message) for one part."""
    n = len(part)
    if n < 3:
        return False, f"only {n} vertex(es); need ≥ 3"
    if n > B2_MAX_VERTICES:
        return False, f"{n} vertices; max is {B2_MAX_VERTICES}"
    if not is_convex(sort_ccw(part)):
        return False, "non-convex"
    return True, f"{n} vertices, convex ✓"


In [ ]:
class PolygonDesigner:
    PICK_RADIUS = 0.005  # m — click tolerance for picking a vertex

    def __init__(self, chassis_side=0.060):
        self.chassis_side = chassis_side
        self.parts = [[]]            # list[list[(x, y)]]
        self.active = 0              # index into self.parts
        self.drag = None             # (part_idx, vertex_idx) or None

        # Figure
        self.fig, self.ax = plt.subplots(figsize=(7, 7))
        self.fig.canvas.toolbar_visible = False
        self.fig.canvas.header_visible = False
        self.fig.canvas.footer_visible = False
        self.fig.canvas.resizable = False

        view = 3 * chassis_side
        self.ax.set_xlim(-view, view)
        self.ax.set_ylim(-view, view)
        self.ax.set_aspect("equal")
        self.ax.grid(True, alpha=0.3)
        self.ax.set_xlabel("body x [m]  →  forward")
        self.ax.set_ylabel("body y [m]  →  left")

        # Mouse events
        self.fig.canvas.mpl_connect("button_press_event",   self._on_press)
        self.fig.canvas.mpl_connect("button_release_event", self._on_release)
        self.fig.canvas.mpl_connect("motion_notify_event",  self._on_motion)

        # Controls
        self.part_select  = widgets.Dropdown(description="Active part",
                                              options=[("part 0 (0 vtx)", 0)],
                                              value=0,
                                              layout=widgets.Layout(width="220px"))
        self.add_btn      = widgets.Button(description="+ part", button_style="success")
        self.del_btn      = widgets.Button(description="delete part", button_style="warning")
        self.name_box     = widgets.Text(value="my_manipulator", description="Name", layout=widgets.Layout(width="320px"))
        self.save_btn     = widgets.Button(description="Save", button_style="primary")
        self.load_dd      = widgets.Dropdown(description="Load", layout=widgets.Layout(width="320px"))
        self.status_html  = widgets.HTML(value="")

        self.part_select.observe(self._on_part_change, names="value")
        self.add_btn.on_click(lambda _: self._add_part())
        self.del_btn.on_click(lambda _: self._delete_part())
        self.save_btn.on_click(lambda _: self._save())
        self.load_dd.observe(self._on_load_change, names="value")

        controls = widgets.VBox([
            widgets.HBox([self.part_select, self.add_btn, self.del_btn]),
            widgets.HBox([self.name_box, self.save_btn]),
            self.load_dd,
            self.status_html,
        ])
        display(controls)

        self._refresh_load_dd()
        self._refresh_part_dd()
        self.redraw()

    # ---- mouse handlers ----

    def _on_press(self, ev):
        if ev.inaxes != self.ax or ev.xdata is None:
            return
        x, y = ev.xdata, ev.ydata
        picked = self._pick(x, y)
        if ev.button == 1:  # left
            if picked is not None:
                self.drag = picked
            else:
                self.parts[self.active].append((x, y))
                self.redraw()
        elif ev.button == 3 and picked is not None:  # right click on vertex
            pi, vi = picked
            del self.parts[pi][vi]
            self.redraw()

    def _on_motion(self, ev):
        if self.drag is None or ev.inaxes != self.ax or ev.xdata is None:
            return
        pi, vi = self.drag
        self.parts[pi][vi] = (ev.xdata, ev.ydata)
        self.redraw()

    def _on_release(self, ev):
        self.drag = None

    def _pick(self, x, y):
        for pi, part in enumerate(self.parts):
            for vi, (vx, vy) in enumerate(part):
                if (vx - x) ** 2 + (vy - y) ** 2 < self.PICK_RADIUS ** 2:
                    return (pi, vi)
        return None

    # ---- part management ----

    def _add_part(self):
        self.parts.append([])
        self.active = len(self.parts) - 1
        self._refresh_part_dd()
        self.redraw()

    def _delete_part(self):
        if len(self.parts) <= 1:
            self.parts = [[]]
            self.active = 0
        else:
            del self.parts[self.active]
            self.active = max(0, self.active - 1)
        self._refresh_part_dd()
        self.redraw()

    def _on_part_change(self, change):
        if change["new"] is None:
            return
        self.active = change["new"]
        self.redraw()

    def _refresh_part_dd(self):
        new_opts = [(f"part {i} ({len(p)} vtx)", i) for i, p in enumerate(self.parts)]
        self.part_select.options = new_opts
        if 0 <= self.active < len(self.parts):
            self.part_select.value = self.active

    # ---- save / load ----

    def _save(self):
        sorted_parts = [sort_ccw(p) for p in self.parts]
        # Validate every part
        problems = []
        for i, p in enumerate(sorted_parts):
            ok, msg = part_status(p)
            if not ok:
                problems.append(f"part {i}: {msg}")
        if problems:
            self.status_html.value = "<b style=\"color:#b00\">Cannot save:</b><br>" + "<br>".join(problems)
            return
        name = self.name_box.value.strip()
        if not name:
            self.status_html.value = "<b style=\"color:#b00\">Need a name.</b>"
            return
        out = {
            "name": name,
            "parts": [[list(v) for v in p] for p in sorted_parts],
        }
        path = CONFIGS_DIR / f"{name}.json"
        path.write_text(json.dumps(out, indent=2) + "\n")
        self.status_html.value = f"<b style=\"color:#0a0\">Saved</b> {path.relative_to(atomsim_root)}"
        self._refresh_load_dd()

    def _refresh_load_dd(self):
        files = sorted(CONFIGS_DIR.glob("*.json"))
        self.load_dd.options = [("— pick a file —", None)] + [(f.stem, f) for f in files]
        self.load_dd.value = None

    def _on_load_change(self, change):
        path = change["new"]
        if path is None:
            return
        data = json.loads(Path(path).read_text())
        self.parts = [[tuple(v) for v in part] for part in data.get("parts", [])]
        if not self.parts:
            self.parts = [[]]
        self.active = 0
        self.name_box.value = data.get("name", path.stem)
        self._refresh_part_dd()
        self.redraw()

    # ---- drawing ----

    def redraw(self):
        self.ax.clear()
        view = 3 * self.chassis_side
        self.ax.set_xlim(-view, view)
        self.ax.set_ylim(-view, view)
        self.ax.set_aspect("equal")
        self.ax.grid(True, alpha=0.3)
        self.ax.set_xlabel("body x [m]  →  forward")
        self.ax.set_ylabel("body y [m]  →  left")

        # Chassis
        s = self.chassis_side
        self.ax.add_patch(Rectangle((-s/2, -s/2), s, s, fill=True, alpha=0.15, edgecolor="black", facecolor="gray"))
        # Forward arrow
        self.ax.annotate("", xy=(s, 0), xytext=(0, 0),
                         arrowprops=dict(arrowstyle="->", color="black", alpha=0.5))

        statuses = []
        for i, part in enumerate(self.parts):
            color = PART_COLORS[i % len(PART_COLORS)]
            sorted_p = sort_ccw(part)
            ok, msg = part_status(sorted_p)
            outline_color = color if ok else "red"
            if len(sorted_p) >= 3:
                self.ax.add_patch(MplPolygon(sorted_p, closed=True, fill=True, alpha=0.3,
                                             edgecolor=outline_color, facecolor=color, linewidth=2))
            elif len(sorted_p) == 2:
                xs, ys = zip(*sorted_p)
                self.ax.plot(xs, ys, color=outline_color, linewidth=2)
            for j, (vx, vy) in enumerate(sorted_p):
                marker = "o" if i == self.active else "."
                self.ax.plot(vx, vy, marker, color=color, markersize=10 if i == self.active else 7,
                             markeredgecolor="black", markeredgewidth=0.5)
            statuses.append(f"<b style=\"color:{color}\">part {i}</b> — {msg}")

        active_marker = f" (active: part {self.active})" if self.parts else ""
        self.status_html.value = "<br>".join(statuses) + f"<br><i>chassis_side = {self.chassis_side} m{active_marker}</i>"
        self.fig.canvas.draw_idle()


In [ ]:
# Instantiate the designer. Adjust chassis_side as you like.
designer = PolygonDesigner(chassis_side=0.06)


## After saving

Files land in `AtomSim/sim/configs/manipulators/<name>.json`. To use a saved manipulator from the simulator side, reference it from a robot config (e.g., `AtomSim/sim/configs/robots/diff_drive_default.json`):

```json
{
  "name": "diff_drive_default",
  "chassis_side": 0.10,
  "manipulator": "my_manipulator"
}
```

The sim loader will look up `"manipulator"` by name in the manipulators directory. Pick a different file there to swap shapes without touching code.